# CAID Benchmark — Google Colab Runner

Run the CAID Benchmark on Google's free infrastructure. No VPN, no payment, no setup.

**Repository:** https://github.com/revenue7-eng/caid-benchmark

## Steps
1. Run cell 1 — clone the repo and install dependencies (~30 seconds)
2. Run cell 2 — set your API keys (interactive input, won't be saved in notebook)
3. Run cell 3 — smoke test (~10 minutes, validates everything works)
4. Run cell 4 — full benchmark (4-8 hours; survives Colab timeouts via Google Drive)
5. Run cell 5 — generate report and download results

**Important:** Colab free tier disconnects after ~12 hours of idle / 90 min inactive UI. Cell 4 mounts Google Drive and writes there — if you disconnect mid-run, you can resume from Drive.

## Cell 1 — Clone repo and install dependencies

In [ ]:
!git clone https://github.com/revenue7-eng/caid-benchmark.git /content/caid-benchmark
%cd /content/caid-benchmark
!pip install -q -r requirements.txt
!chmod +x run_full_pipeline.sh

# Verify classifier works
!python src/test_classifier.py

Expected output at the bottom: `Passed: 12/12 Failed: 0 Ambiguous: 0`. If anything else — stop and check.

## Cell 2 — Set API keys

Keys are entered via password-input (hidden), stored in environment variables for the current session only. They won't be saved in the notebook file.

Set only the keys you have. Skip the rest by pressing Enter (leaves empty).

In [ ]:
import os
from getpass import getpass

providers = [
    ('GROQ_API_KEY', 'Groq', 'gsk_...'),
    ('OPENROUTER_API_KEY', 'OpenRouter', 'sk-or-v1-...'),
    ('CEREBRAS_API_KEY', 'Cerebras (optional)', 'csk-...'),
    ('SAMBANOVA_API_KEY', 'SambaNova (optional)', '...'),
    ('MISTRAL_API_KEY', 'Mistral (optional)', '...'),
    ('GOOGLE_API_KEY', 'Google AI Studio (optional)', 'AIza...'),
    ('HF_TOKEN', 'HuggingFace (optional)', 'hf_...'),
    ('ANTHROPIC_API_KEY', 'Anthropic for LLM-judge (optional, paid)', 'sk-ant-...'),
]

for env_var, name, hint in providers:
    val = getpass(f'{name} ({hint}) — Enter to skip: ').strip()
    if val:
        os.environ[env_var] = val
        print(f'  {name}: set ({val[:10]}...)')
    else:
        print(f'  {name}: skipped')

print()
print('Active providers:', [k for k in [
    'GROQ_API_KEY', 'OPENROUTER_API_KEY', 'CEREBRAS_API_KEY',
    'SAMBANOVA_API_KEY', 'MISTRAL_API_KEY', 'GOOGLE_API_KEY', 'HF_TOKEN'
] if os.environ.get(k)])

## Cell 3 — Smoke test

Quick validation: 1 model per provider × N=2 replicates. Should take 5–15 minutes depending on how many providers you set up.

If smoke test fails — check the error and the providers you connected. Don't proceed to full run until smoke passes.

In [ ]:
%cd /content/caid-benchmark
!./run_full_pipeline.sh --smoke

## Cell 4 — Full benchmark with Google Drive backup

Mounts your Google Drive and writes all benchmark data there. This way:
- If Colab disconnects (idle timeout, network issue, browser crash) — data is preserved on Drive
- You can resume the run from where it stopped
- Results are accessible from anywhere via your Drive

First run mounts Drive (you'll be asked to authenticate via Google account). Subsequent runs reuse the mount.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create persistent data directory on Drive
import os
DRIVE_DATA = '/content/drive/MyDrive/caid_benchmark_data'
os.makedirs(DRIVE_DATA, exist_ok=True)

# Symlink so the script writes there
%cd /content/caid-benchmark
!rm -rf data
!ln -s {DRIVE_DATA} data

print(f'Data will be saved to Google Drive: {DRIVE_DATA}')
!ls -la data

Now launch the full run. **Safe mode** is recommended: pacing 4-6s, chunks of 40 with 5min pauses, randomised model order. Lower block risk, longer wall time.

Total expected time: 4-12 hours depending on number of providers connected.

**To prevent Colab idle disconnect:** open browser DevTools (F12), Console tab, paste:
```javascript
function ClickConnect(){console.log('Anti-idle');document.querySelector('colab-connect-button').shadowRoot.getElementById('connect').click()}
setInterval(ClickConnect,60000)
```
This sends a fake click every 60s to keep Colab from going idle.

In [ ]:
%cd /content/caid-benchmark
!./run_full_pipeline.sh --safe

### Resume after disconnect

If Colab disconnects mid-run:
1. Reconnect runtime
2. Re-run Cell 1 (clone) and Cell 2 (keys)
3. Re-run the Drive mount cell above
4. Find your run_id from the previous run:

In [ ]:
# List previous runs on Drive
!ls -lat /content/drive/MyDrive/caid_benchmark_data/raw/ | head

In [ ]:
# Resume — replace RUN_ID with the directory name from the listing above
RUN_ID = '20260426_xxxxxx_xxxxxx'  # <-- edit this

%cd /content/caid-benchmark
!./run_full_pipeline.sh --resume {RUN_ID}

## Cell 5 — Inspect results and download

After the run completes, results are in `/content/drive/MyDrive/caid_benchmark_data/raw/<RUN_ID>/`.

In [ ]:
# List all runs
import os
runs_dir = '/content/drive/MyDrive/caid_benchmark_data/raw'
if os.path.exists(runs_dir):
    for r in sorted(os.listdir(runs_dir)):
        path = os.path.join(runs_dir, r)
        files = os.listdir(path) if os.path.isdir(path) else []
        print(f'  {r}  ({len(files)} files)')
else:
    print('No runs yet')

In [ ]:
# View summary table for a specific run
RUN_ID = '20260426_xxxxxx_xxxxxx'  # <-- edit

import pandas as pd
csv_path = f'/content/drive/MyDrive/caid_benchmark_data/raw/{RUN_ID}/metrics_per_model.csv'
df = pd.read_csv(csv_path)
df.sort_values('overall_rate', ascending=False)

In [ ]:
# Download specific files to your computer
from google.colab import files
RUN_ID = '20260426_xxxxxx_xxxxxx'  # <-- edit
base = f'/content/drive/MyDrive/caid_benchmark_data/raw/{RUN_ID}'

for fname in ['metrics_per_model.csv', 'metrics_cells.csv', 'metrics.json',
              'classifications.jsonl']:
    fpath = f'{base}/{fname}'
    if os.path.exists(fpath):
        files.download(fpath)

---

## Tips

- **Free tier limits.** Colab gives ~12 hours/day of compute. Full CAID run on all 7 providers can take 12-18 hours, so plan to span 2 days. Resume picks up where you stopped.

- **Idle disconnect.** Colab disconnects after 90min of UI inactivity. Either keep the tab focused, or use the JavaScript anti-idle snippet from Cell 4 instructions.

- **Network.** Colab runs from US data centres — providers (Groq, Cerebras) won't geo-block.

- **Costs.** This notebook uses only free tier. No GPU needed (we just call APIs). Compute time is minimal.

- **Privacy.** API keys are typed via getpass (hidden) and held in os.environ for the runtime only. They are NOT saved in the notebook file. Do not commit this notebook with keys filled in.